# Lecture 1: Intro to harness

## The big idea

An LLM is a function that takes **text in** and returns **text out**. That's it.

It cannot run code, call an API, read a file or press a button. When you see an "AI agent" calculate, search the web or edit code, the model isn't doing that. **Ordinary code around the model** is doing it. That code is called the **harness**.

The harness does four things:

1. **Describes the tools** to the model, using the system prompt.
2. **Reads** the text the model returns.
3. **Detects** whether that text is a request to call a tool. If it is, the harness **runs the real function**.
4. **Sends the result back** to the model, so it can write the final answer.

```
 user question
      │
      ▼
   ┌─────┐   text    ┌───────────────── harness (plain Python) ─────────────────┐
   │ LLM │ ────────► │ parse ──► tool call? ──yes──► run the real function(s)   │
   └─────┘           │               │                         │                │
      ▲              │               no                        ▼                │
      │              │               │                  "TOOL RESULT ..."       │
      │              └───────────────┼─────────────────────────┼────────────────┘
      │                              ▼                         │
      │                        final answer                    │
      └────────────────────────────────────────────────────────┘
```

In this notebook we build all of this from scratch for a **calculator** with 4 tools: `add`, `subtract`, `multiply` and `divide`.

We deliberately **don't** use the API's built-in tool-calling feature. We make the model write JSON as plain text, and parse it ourselves. That's exactly what built-in tool calling does for you behind the scenes.

## Step 0: Setup

You need an Anthropic API key in the `ANTHROPIC_API_KEY` environment variable.

In [ ]:
%pip install -q anthropic

In [ ]:
import json
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-opus-5-5"

## Step 1: The tools are just Python functions

Nothing special here. These are normal functions. The model will never "touch" them. Only our harness calls them.

In [ ]:
def add(a, b):
    return a + b


def subtract(a, b):
    return a - b


def multiply(a, b):
    return a * b


def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


# The registry: tool name (what the model writes) -> real Python function
TOOLS = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
}

## Step 2: The system prompt tells the model the "protocol"

The model doesn't know our tools exist unless we tell it. We also have to agree on a **format** the harness can recognize. Here, a tool call is a JSON list of calls:

```json
[{"tool": "multiply", "arguments": {"a": 6, "b": 7}}]
```

Because it's a list, the model can ask for **several tools at once** (parallel tool calls) when it needs independent calculations.

When the harness has run the tools, it sends the results back in a message that starts with `TOOL RESULT`.

In [ ]:
SYSTEM_PROMPT = """You are a calculator assistant. You are bad at arithmetic, so you must
never calculate anything yourself. Always use the tools below.

Available tools:
- add(a, b)       returns a + b
- subtract(a, b)  returns a - b
- multiply(a, b)  returns a * b
- divide(a, b)    returns a / b

To call tools, reply with ONLY a JSON list of tool calls in exactly this format, and nothing else:
[{"tool": "<tool name>", "arguments": {"a": <number>, "b": <number>}}]

If the question needs several independent calculations, put all of them in the
same list so they can run at the same time.

After calling tools you will receive a message like:
TOOL RESULT (<tool name>): <value>

Then reply with the final answer in plain English, with no JSON."""

## Step 3: Calling the LLM is text in, text out

This function sends the conversation to the model and returns the text it wrote. Notice the return type: a **string**. The model can't return anything else.

In [ ]:
def call_llm(messages):
    """Send the conversation to the model and return its reply as plain text."""
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=16000,
        system=SYSTEM_PROMPT,
        messages=messages,
        # If a safety classifier declines, retry on a fallback model automatically.
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    if response.stop_reason == "refusal":
        raise RuntimeError("The model declined to answer")
    # The response is a list of blocks. We only care about the text ones.
    return "".join(block.text for block in response.content if block.type == "text")

## Step 4: The LLM on its own, with no harness

Let's ask a question and print **exactly** what the model returns.

In [ ]:
reply = call_llm([{"role": "user", "content": "What is 48219 * 3172?"}])

print(type(reply))
print(reply)

### What just happened?

The model wrote something like `[{"tool": "multiply", "arguments": {"a": 48219, "b": 3172}}]`.

But **nothing was multiplied**. The model produced a string that *looks like* a request. Nobody was listening for it, so nothing happened.

> The LLM has no capabilities of its own. It can only *ask*. Something has to read the request and act on it. That something is the harness.

## Step 5: The harness, part 1: parsing the model's output

This is the heart of the lecture. We read the text and decide: **is this a tool request, or a final answer?**

The parser returns a list of tool calls. An empty list means "no tool calls, this is the final answer".

Models aren't perfectly consistent, so we're a little forgiving. We also accept a single JSON object instead of a list, and JSON wrapped in a ```` ```json ```` code fence or with some text around it.

In [ ]:
def parse_tool_calls(text):
    """Return a list of {"tool": ..., "arguments": {...}} found in the text (empty if none)."""
    starts = [i for i in (text.find("["), text.find("{")) if i != -1]
    if not starts:
        return []  # no JSON at all -> it's a normal answer

    try:
        # raw_decode reads one JSON value and ignores whatever comes after it
        data, _ = json.JSONDecoder().raw_decode(text[min(starts):])
    except json.JSONDecodeError:
        return []  # looked like JSON but wasn't valid

    if isinstance(data, dict):
        data = [data]  # a single call without the list around it
    if not isinstance(data, list):
        return []

    return [
        {"tool": item["tool"], "arguments": item.get("arguments", {})}
        for item in data
        if isinstance(item, dict) and "tool" in item
    ]

Let's test the parser on some hand-written strings. **No LLM is involved here.** The parser is just string handling.

In [ ]:
examples = [
    '[{"tool": "add", "arguments": {"a": 2, "b": 3}}]',
    '[{"tool": "add", "arguments": {"a": 1, "b": 2}}, {"tool": "multiply", "arguments": {"a": 3, "b": 4}}]',
    '```json\n[{"tool": "divide", "arguments": {"a": 10, "b": 4}}]\n```',
    '{"tool": "subtract", "arguments": {"a": 9, "b": 5}}',
    "The answer is 42.",
    '[{"tool": "multiply", "arguments": {"a": 2,',  # broken JSON
]

for text in examples:
    print(repr(text))
    print("  ->", parse_tool_calls(text))

## Step 6: The harness, part 2: actually running the tool

The model wrote a **name** and some **arguments**. We look the name up in our `TOOLS` registry and call the real function.

If something goes wrong (an unknown tool, bad arguments, dividing by zero), we don't crash. We send the error back to the model as text, so it can explain what happened.

In [ ]:
def execute_tool(call):
    """Run the requested tool and return the result as a string for the model."""
    name = call["tool"]
    args = call["arguments"]

    if name not in TOOLS:
        return f"ERROR: unknown tool '{name}'. Available tools: {', '.join(TOOLS)}"

    try:
        result = TOOLS[name](**args)
    except Exception as e:
        return f"ERROR: {e}"

    return str(result)


print(execute_tool({"tool": "multiply", "arguments": {"a": 48219, "b": 3172}}))
print(execute_tool({"tool": "divide", "arguments": {"a": 1, "b": 0}}))
print(execute_tool({"tool": "sqrt", "arguments": {"a": 9}}))

## Step 7: Putting the harness together

`ask` runs one full round trip and prints every stage, so you can see what the model writes and what the harness does with it:

1. Send the question to the LLM.
2. Parse the reply. If there are no tool calls, the reply is already the answer.
3. Run **every** tool call the model asked for.
4. Send all the results back in one message, and print the model's final answer.

In [ ]:
def ask(question):
    messages = [{"role": "user", "content": question}]

    # 1. The LLM sees the question and writes some text
    reply = call_llm(messages)
    print(f"LLM wrote:      {reply}")

    # 2. The harness checks whether that text is a tool request
    calls = parse_tool_calls(reply)
    if not calls:
        print("No tool calls, so this is the final answer.")
        return reply

    # 3. The harness runs each requested tool
    results = []
    for call in calls:
        result = execute_tool(call)
        print(f"Harness ran:    {call['tool']}({call['arguments']}) = {result}")
        results.append(f"TOOL RESULT ({call['tool']}): {result}")

    # 4. The harness sends the results back, and the LLM writes the answer
    messages.append({"role": "assistant", "content": reply})
    messages.append({"role": "user", "content": "\n".join(results)})
    answer = call_llm(messages)
    print(f"Final answer:   {answer}")
    return answer

## Example 1: A single tool call (add)

In [ ]:
ask("What is 1234 + 5678?")

## Example 2: A single tool call (multiply)

The same question we asked in Step 4. Then nothing happened. Now there's a harness listening.

In [ ]:
ask("What is 48219 * 3172?")

## Example 3: Parallel tool calls

Three calculations that don't depend on each other. The model asks for **all three tools in one reply**, the harness runs all three, and sends all the results back together.

This takes one extra round trip to the LLM instead of three. Here the harness runs the tools one after another, but since they're independent, a real harness could run them at the same time.

In [ ]:
ask("What are 125 + 375, 9876 - 5432 and 256 * 64?")

## Key takeaways

- **The LLM only produces text.** A "tool call" is just text in a format we agreed on.
- **The harness gives the model its abilities.** It parses the text, runs real code and reports back.
- **The system prompt is the contract** between the model and the harness: which tools exist, and what a call looks like.
- **Parallel tool calls** are just several calls in one reply. The harness runs them all and returns all the results at once.

### How this maps to real APIs

Modern LLM APIs have **built-in tool calling**. You pass a list of tools with JSON schemas, and the response comes back with structured `tool_use` blocks instead of free text. That's the same idea we built by hand today. The provider writes the "describe the tools" part of the prompt and does the parsing for you, so it's more reliable. **Running the tool and sending back the result is still your harness's job.**

### Exercises

1. Add a `power(a, b)` tool. What do you need to change? (Hint: two places.)
2. Ask a question that needs a tool that doesn't exist, like a square root. What does the model do?
3. Remove the line "never calculate anything yourself" from the system prompt. Does the model still use the tools?
4. Ask `ask("Add 12 and 30, then multiply the result by 4.")`. The second calculation needs the first one's result, so it can't run in parallel. What goes wrong, and how would you change `ask` to fix it?